In [ ]:
# pip install some library
%pip install einops matplotlib jupyter_black

In [ ]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from einops import rearrange, reduce
from unet import SRUNet

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import s3fs

# Code to connect to share data in S3
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": S3_ENDPOINT_URL})
# now you should be able to see my share repository "mfauvel/diffusion/UNET_DATA
# Wae are going to copy locally the data set
for rep in ["gt", "images"]:
    data_dir = Path.home() / f"UNET_DATA/{rep}"
    data_dir.mkdir(exist_ok=True, parents=True)
    fs.get(f"s3://mfauvel/diffusion/UNET_DATA/{rep}/", data_dir, recursive=True)

# Data Preparation

In [ ]:
# create torch data set
class PatchDataSet(Dataset):
    def __init__(self, path: Path, dtype: torch.dtype = torch.bfloat16):
        self.train_list = list((path / "images").glob("*.tif"))
        self.gt_list = list((path / "gt").glob("*.tif"))
        self.dtype = dtype

    def __len__(self):
        return len(self.train_list)

    def __getitem__(self, idx: int) -> [torch.Tensor, torch.Tensor]:
        # Read image and gt
        image = rearrange(
            torch.as_tensor(
                np.array(Image.open(self.train_list[idx])), dtype=self.dtype
            ),
            "H W C ->C H W",
        )
        gt = torch.as_tensor(np.array(Image.open(self.gt_list[idx])), dtype=self.dtype)
        return (image, gt)

In [ ]:
class Scaler(torch.nn.Module):
    def __init__(
        self,
        dataset: Dataset,
        mean: list | None = [103.2342, 108.9519, 100.1419],
        std: list | None = [51.3393, 46.7957, 44.9176],
    ):
        super(Scaler, self).__init__()
        # Compute the number of channels
        image, _ = dataset[0]
        n_channels, height, width = image.shape
        dtype = image.dtype
        device = image.device

        # Init Tensor
        if mean == std == None:
            self.mean_ = torch.nn.Parameter(
                torch.zeros((n_channels,), dtype=dtype, device=device),
                requires_grad=False,
            )
            std_ = torch.zeros_like(self.mean_)

            # Compute Mean
            for image, _ in dataset:
                self.mean_ += reduce(image, "C H W -> C", "mean")
            self.mean_ /= len(dataset)

            # Compute Std
            for image, _ in dataset:
                std_ += reduce(
                    (self.mean_[:, None, None] - image) ** 2, "C H W -> C", "mean"
                )
            std_ /= len(dataset)

            self.std_ = torch.nn.Parameter(torch.sqrt(std_), requires_grad=False)
            self.std_[self.std_ == 0] = 1
        else:
            assert (mean is not None) and (std is not None)
            self.mean_ = torch.nn.Parameter(
                torch.as_tensor(mean, dtype=dtype, device=device).reshape(
                    (n_channels, 1, 1)
                ),
                requires_grad=False,
            )
            self.std_ = torch.nn.Parameter(
                torch.as_tensor(std, dtype=dtype, device=device).reshape(
                    (n_channels, 1, 1)
                ),
                requires_grad=False,
            )

    def __call__(self, X: torch.Tensor) -> torch.Tensor:
        assert X.shape[0] == self.mean_.shape[0]
        return (X - self.mean_) / self.std_

    def inverse_transform(self, X: torch.Tensor) -> torch.Tensor:
        assert X.shape[0] == self.mean_.shape[0]
        return X * self.std_ + self.mean_


class DatasetTransformer(torch.utils.data.Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __getitem__(self, index):
        img, target = self.base_dataset[index]
        return self.transform(img), target / 255.0

    def __len__(self):
        return len(self.base_dataset)

In [ ]:
# Initialize data set and scaler
dtype = torch.bfloat16
data_dir = Path.home() / "UNET_DATA"
dataset = PatchDataSet(data_dir, dtype=dtype)
scaler = Scaler(dataset)
scale_dataset = DatasetTransformer(dataset, scaler)
train_dataset, valid_dataset, test_dataset = torch.utils.data.random_split(
    scale_dataset, [0.1, 0.1, 0.8]
)

In [ ]:
# Prepare data loader
num_threads = 16  # Loading the dataset is using 4 CPU threads
batch_size = 16  # Using minibatches of 512 samples

train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_threads
)

valid_loader = torch.utils.data.DataLoader(
    dataset=valid_dataset, batch_size=batch_size, shuffle=False, num_workers=num_threads
)

test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_threads
)

In [ ]:
# Init model
model = SRUNet(in_channels=3, out_channels=1, depth=3, dropout=0.25)
model.to(dtype=dtype)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_epochs = 5
learning_rate = 0.001
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
train_set_len = len(train_loader)
val_set_len = len(valid_loader)
train_loss, val_loss = [], []
best_loss = 10000000.0
model.to(device)

for epoch in range(n_epochs):
    model.train()
    accu = 0.0

    for X_, y_ in train_loader:
        X_ = X_.to(device)
        y_ = y_.to(device)
        # Forward pass
        y_hat = model(X_)
        loss = loss_fn(y_hat, y_.unsqueeze(1))
        accu += loss.item()

        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss.append(accu / train_set_len)

    # Validation - no gradient & eval mode
    model.eval()
    accu = 0.0
    with torch.no_grad():
        for X_, y_ in valid_loader:
            X_ = X_.to(device)
            y_ = y_.to(device)
            # Forward pass
            y_hat = model(X_)
            loss = loss_fn(y_hat, y_.unsqueeze(1))
            accu += loss.item()
        val_loss.append(accu / val_set_len)
        if best_loss > val_loss[-1]:
            best_loss = val_loss[-1]
            torch.save(model.state_dict(), "best_model.pt")
    print(f"Train_loss: {train_loss[-1]} and Val_loss: {val_loss[-1]}")